In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Change this if your CNEEP_v2 folder lives elsewhere in Google Drive.
drive_path = '/content/drive/MyDrive/CNEEP_v2'
if os.path.exists(drive_path):
    os.chdir(drive_path)
    print(f'Changed working directory to {os.getcwd()}')
else:
    print(f'Directory {drive_path} not found. Please verify the path.')


# ShellForce CNEEP for AMB2D Circle Cluster

Colab notebook for Active Model B 2D circle-cluster trajectories using the ShellForce short-time CNEEP model.

In [ ]:
import sys
import os
CNEEP_V2_ROOT = os.path.abspath('/content/drive/MyDrive/CNEEP_v2')
if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'data', 'AMB'))

from argparse import Namespace
import numpy as np
import torch
from datetime import datetime
from IPython.display import display
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import linregress
from sklearn.metrics import r2_score
from tqdm import tqdm
import matplotlib.pyplot as plt

from utils.sampler import CartesianSeqSampler
from generate_trajectories import ActiveModelB


In [ ]:
# Hyper parameters
opt = Namespace()
opt.model_type = 'MultiScaleShellForceCNEEP2D'  # 'MultiScaleShellForceCNEEP2D' or 'MultiScaleShellForceCNEEP2D_Tanh'
opt.device = 'cuda' if torch.cuda.is_available() else 'cpu'
opt.alpha = -0.5
opt.lam = 0.0
opt.periodic = True
opt.positional = False
opt.n_components = 1
opt.n_iter = 5000
opt.train_batch_size = 2048
opt.test_batch_size = 2048
opt.video_batch_size = 256
opt.lr = 3e-4
opt.wd = 0.0
opt.input_scalar = 1
opt.loss_scalar = 1
opt.scalar = 1
opt.clip_norm = 1
opt.max_distance = 3
opt.include_k0 = True
opt.beta = 0.0
opt.shell_center_mode = 'add'  # 'relative_only', 'gated', or 'add'
opt.shell_force_bias = True
opt.shell_relative_mode = 'learned_absolute'  # 'fixed_sum'/'relative_sum', 'absolute_sum', 'learned', 'learned_absolute', 'learned_sum', or 'learned_absolute_sum'
opt.shell_weight_normalization = 'none'  # 'none' for sum_delta, 'mean' for shell average
opt.shell_force_activation = 'elu'  # 'elu', 'relu', 'tanh', or 'identity'
opt.record_freq = 100
opt.seed = 3
opt.n_layer = 2
opt.n_channel = 64
opt.n_hidden = 3
opt.input_shape = (64, 64)
opt.M = 256
opt.M_test = 1
opt.L = 1000
opt.L_test = 5000
opt.seq_len = 2
opt.val_ratio = 0.2
opt.time_step = 0.001

# Active Model B 2D parameters
amb_kwargs = dict(
    Lx=64, Ly=64, dx=1.0,
    a=0.25, b=0.25, kappa=4.0,
    lam=1.0, D=0.1, dt=0.001,
    smooth=True,
    backend='torch',
    use_gpu=True,
    epr_mode='ours',
    epr_mu_active_only=True,
)
n_steps = opt.L
burn_in = 10000
init_mode = 'circle'
dt_eff = amb_kwargs['dt']
vol = amb_kwargs['Lx'] * amb_kwargs['Ly'] * amb_kwargs['dx']**2

torch.manual_seed(opt.seed)
np.random.seed(opt.seed)
result_folder = os.path.join(CNEEP_V2_ROOT, 'results')
current_result_folder = os.path.join(result_folder, f'CorrAMB2D-ShellForce-{datetime.now().strftime("%Y-%m-%d-%H%M%S")}')
os.makedirs(current_result_folder, exist_ok=True)
current_checkpoint_path = os.path.join(current_result_folder, 'model_parameter.pth.tar')
best_checkpoint_path = os.path.join(current_result_folder, 'best_model_parameter.pth.tar')
print(f'Device: {opt.device}')
print(f'Results: {current_result_folder}')


In [ ]:
print(f'[INFO] Generating TRAIN circle-cluster trajectories (M={opt.M}, L={opt.L}) on {opt.device}...')
train_seed = 42
np.random.seed(train_seed)
torch.manual_seed(train_seed)
model_amb_train = ActiveModelB(**amb_kwargs)
trajectories_train = model_amb_train.generate_trajectories(
    n_trajectories=opt.M,
    n_steps=opt.L,
    burn_in=burn_in,
    init_mode=init_mode,
    show_progress=True,
)
print('Train shape:', trajectories_train.shape)


In [ ]:
print(f'[INFO] Generating TEST circle-cluster trajectories (M={opt.M_test}, L={opt.L_test}) on {opt.device}...')
test_seed = 123
np.random.seed(test_seed)
torch.manual_seed(test_seed)
model_amb_test = ActiveModelB(**amb_kwargs)
trajectories_test = model_amb_test.generate_trajectories(
    n_trajectories=opt.M_test,
    n_steps=opt.L_test,
    burn_in=burn_in,
    init_mode=init_mode,
    show_progress=True,
)
traj_test = trajectories_test
print('Test shape:', trajectories_test.shape)


In [ ]:
# Prepare video tensors: (M, L, 1, Lx, Ly)
train_val_split_idx = int(opt.M * (1 - opt.val_ratio))
M_train_new = train_val_split_idx
M_val = opt.M - M_train_new

train_video = torch.from_numpy(trajectories_train[:M_train_new]).float().to(opt.device).unsqueeze(2)
val_video = torch.from_numpy(trajectories_train[M_train_new:]).float().to(opt.device).unsqueeze(2)
test_video = torch.from_numpy(trajectories_test).float().to(opt.device).unsqueeze(2)

print('Train video:', train_video.shape)
print('Val video:', val_video.shape)
print('Test video:', test_video.shape)


In [ ]:
from models.NEEP_ShellForce_2D import MultiScaleShellForceCNEEP2D, MultiScaleShellForceCNEEP2D_Tanh

mean = torch.mean(train_video)
std = torch.std(train_video)
transform = lambda x: (x - mean) * opt.input_scalar / std

if opt.model_type == 'MultiScaleShellForceCNEEP2D_Tanh':
    model = MultiScaleShellForceCNEEP2D_Tanh(opt).to(opt.device)
elif opt.model_type == 'MultiScaleShellForceCNEEP2D':
    model = MultiScaleShellForceCNEEP2D(opt).to(opt.device)
else:
    raise ValueError(f'Unknown model_type: {opt.model_type}')

optim = torch.optim.AdamW(model.parameters(), opt.lr, weight_decay=opt.wd)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

train_sampler = CartesianSeqSampler(M_train_new, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device)
val_sampler = CartesianSeqSampler(M_val, opt.L, opt.seq_len, opt.test_batch_size, device=opt.device, train=False)

smoothing = 0.5
smooth_train_loss = None
smooth_val_loss = None
best_val_loss = float('inf')
train_losses = []
valid_losses = []
history_iters = []
history_train_loss = []
history_val_loss = []

fig, ax = plt.subplots(figsize=(8, 5))
line_train, = ax.plot([], [], label='train_loss')
line_val, = ax.plot([], [], label='val_loss')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.legend()
display_handle = display(fig, display_id=True)
plt.close(fig)

for it in tqdm(range(1, opt.n_iter + 1)):
    model.train()
    batch = next(train_sampler)
    b0 = batch[0].to(train_video.device)
    slices = [train_video[(b0, batch[1][i].to(train_video.device))] for i in range(opt.seq_len)]
    x = transform(torch.cat(slices, dim=1).float().to(opt.device))

    J_all = model(x) / opt.scalar
    ep_density = J_all.sum(dim=1)

    optim.zero_grad()
    if opt.alpha == 0:
        loss = (- ep_density + (torch.exp(-ep_density) - 1)).mean()
    else:
        loss = (- (torch.exp(opt.alpha * ep_density) - 1) / opt.alpha
                + (torch.exp(-(1 + opt.alpha) * ep_density) - 1) / (1 + opt.alpha)).mean()
    (loss * opt.loss_scalar).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=opt.clip_norm)
    optim.step()
    train_losses.append(loss.item())

    if it % opt.record_freq == 0 or it == 1:
        model.eval()
        val_loss_acc = 0.0
        n_val = 0
        with torch.no_grad():
            for vb in val_sampler:
                vb0 = vb[0].to(val_video.device)
                vslices = [val_video[(vb0, vb[1][i].to(val_video.device))] for i in range(opt.seq_len)]
                vx = transform(torch.cat(vslices, dim=1).float().to(opt.device))
                vJ = model(vx) / opt.scalar
                v_ep_density = vJ.sum(dim=1)
                if opt.alpha == 0:
                    vloss = (- v_ep_density + (torch.exp(-v_ep_density) - 1)).sum().item()
                else:
                    vloss = (- (torch.exp(opt.alpha * v_ep_density) - 1) / opt.alpha
                            + (torch.exp(-(1 + opt.alpha) * v_ep_density) - 1) / (1 + opt.alpha)).sum().item()
                val_loss_acc += vloss
                n_val += vx.shape[0]
        avg_val = val_loss_acc / n_val
        valid_losses.append(avg_val)

        state = {
            'settings': opt.__dict__,
            'state_dict': model.state_dict(),
            'optimizer': optim.state_dict(),
            'iteration': it,
            'mean': mean.detach().cpu(),
            'std': std.detach().cpu(),
        }
        torch.save(state, current_checkpoint_path)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(state, best_checkpoint_path)

        if smooth_train_loss is None:
            smooth_train_loss = loss.item()
            smooth_val_loss = avg_val
        else:
            smooth_train_loss = smoothing * smooth_train_loss + (1 - smoothing) * loss.item()
            smooth_val_loss = smoothing * smooth_val_loss + (1 - smoothing) * avg_val

        history_iters.append(it)
        history_train_loss.append(smooth_train_loss)
        history_val_loss.append(smooth_val_loss)
        line_train.set_data(history_iters, history_train_loss)
        line_val.set_data(history_iters, history_val_loss)
        ax.relim(); ax.autoscale_view()
        display_handle.update(fig)

print('Training finished.')
print(f'Best val loss: {best_val_loss:.6e}')
print(f'Checkpoint saved: {current_checkpoint_path}')


In [ ]:
# Load the best or final model
load_best = True
load_path = best_checkpoint_path if load_best and os.path.exists(best_checkpoint_path) else current_checkpoint_path
checkpoint = torch.load(load_path, map_location=opt.device)
model.load_state_dict(checkpoint['state_dict'])
model.eval()
print(f'Loaded: {load_path}')
print(f"Iteration: {checkpoint.get('iteration', 'Unknown')}")


In [ ]:
# Ground truth EPR rate on TEST data
stride = opt.seq_len - 1
n_windows = (opt.L_test - 1) // stride
gt_total_epr = np.zeros(n_windows)
gt_epr_map_sum = np.zeros((amb_kwargs['Lx'], amb_kwargs['Ly']))

traj_test_gpu = torch.tensor(traj_test, dtype=torch.float64, device=opt.device)
print(f'[INFO] Computing GT EPR on TEST data (n_windows={n_windows})...')
for w in tqdm(range(n_windows)):
    t_start = w * stride
    window_epr_map = np.zeros((amb_kwargs['Lx'], amb_kwargs['Ly']))
    for s in range(stride):
        t = t_start + s
        epr_map = model_amb_test.compute_local_epr_density(traj_test_gpu[:, t], traj_test_gpu[:, t + 1])
        mean_epr_map = epr_map.mean(dim=0)
        window_epr_map += mean_epr_map.detach().cpu().numpy()
    gt_epr_map_sum += window_epr_map
    gt_total_epr[w] = np.sum(window_epr_map) * amb_kwargs['dx']**2

gt_epr_maps = (gt_epr_map_sum / n_windows)[np.newaxis, ...]
print(f'GT mean EPR rate: {gt_total_epr.mean():.6e}')


In [ ]:
model.eval()
pred_total_ep = []
test_sampler = CartesianSeqSampler(opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size, device=opt.device, train=False)
with torch.no_grad():
    for batch in test_sampler:
        b0 = batch[0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.cat(slices, dim=1).float().to(opt.device))
        J_all = model(x) / opt.scalar
        pred_total_ep.append(J_all.sum(dim=1).cpu().numpy())

pred_total_ep = np.concatenate(pred_total_ep) * vol
pred_epr_rate = pred_total_ep / dt_eff
min_len = min(len(gt_total_epr), len(pred_epr_rate))
time_axis = np.arange(min_len) * dt_eff * stride

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
axes[0].plot(time_axis, gt_total_epr[:min_len], lw=0.5, alpha=0.6, label='GT EPR')
axes[0].plot(time_axis, pred_epr_rate[:min_len], lw=0.5, alpha=0.6, label='Pred EPR')
axes[0].set_ylabel('EPR')
axes[0].set_title('Instantaneous EPR Comparison')
axes[0].legend()

axes[1].plot(time_axis, np.cumsum(gt_total_epr[:min_len] * dt_eff), label='GT Cumul EP')
axes[1].plot(time_axis, np.cumsum(pred_total_ep[:min_len]), label='Pred Cumul EP')
axes[1].set_ylabel('Cumulative EP')
axes[1].legend()

window = min(100, max(min_len // 10, 1))
gt_smooth = np.convolve(gt_total_epr[:min_len], np.ones(window) / window, mode='same')
pred_smooth = np.convolve(pred_epr_rate[:min_len], np.ones(window) / window, mode='same')
axes[2].plot(time_axis, gt_smooth, label='GT (Smooth)')
axes[2].plot(time_axis, pred_smooth, label='Pred (Smooth)')
axes[2].set_ylabel('Running Avg')
axes[2].set_xlabel('Time')
axes[2].legend()
plt.tight_layout()
plt.savefig(f'{current_result_folder}/epr_timeseries.png', dpi=150)
plt.show()

print(f'GT mean EPR:   {gt_total_epr[:min_len].mean():.6e}')
print(f'Pred mean EPR: {pred_epr_rate[:min_len].mean():.6e}')


In [ ]:
gt_ep_rate = gt_total_epr[:min_len]
pred_ep_rate = pred_epr_rate[:min_len]
slope, intercept, r_value, p_value, std_err = linregress(gt_ep_rate, pred_ep_rate)
r2 = r2_score(gt_ep_rate, pred_ep_rate)

plt.figure(figsize=(6, 6))
plt.scatter(gt_ep_rate, pred_ep_rate, alpha=0.3, s=2)
plt.plot([gt_ep_rate.min(), gt_ep_rate.max()], [gt_ep_rate.min(), gt_ep_rate.max()], 'k--', label='y = x')
plt.plot(gt_ep_rate, intercept + slope * gt_ep_rate, 'r-', label=f'Fit: R2={r2:.3f}')
plt.xlabel('Ground Truth Total EPR rate')
plt.ylabel('Predicted Total EPR rate')
plt.title('Test Set EPR Scatter Plot')
plt.legend()
plt.tight_layout()
plt.savefig(f'{current_result_folder}/r2_scatter.png', dpi=150)
plt.show()


In [ ]:
model.eval()
test_sampler_one = CartesianSeqSampler(opt.M_test, opt.L_test, opt.seq_len, 1, device=opt.device, train=False)
ens_idx, traj_idx = next(test_sampler_one)
b0 = ens_idx.to(test_video.device)
slices = [test_video[(b0, traj_idx[i].to(test_video.device))] for i in range(opt.seq_len)]
x = transform(torch.cat(slices, dim=1).float().to(opt.device))
with torch.no_grad():
    maps = model(x, return_maps=True) / opt.scalar

phi_t = x[0, 0].detach().cpu().numpy()
pred_map_k = maps[0].detach().cpu().numpy() / (amb_kwargs['dx']**2 * dt_eff)
pred_total_map = pred_map_k.sum(axis=0)
gt_map = gt_epr_maps[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(phi_t.T, origin='lower', cmap='viridis')
axes[0].set_title('Input State phi')
fig.colorbar(im0, ax=axes[0])
vmax = max(np.abs(gt_map).max(), np.abs(pred_total_map).max(), 1e-12)
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
im1 = axes[1].imshow(gt_map.T, origin='lower', cmap='RdBu_r', norm=norm)
axes[1].set_title('GT EPR Map')
fig.colorbar(im1, ax=axes[1])
im2 = axes[2].imshow(pred_total_map.T, origin='lower', cmap='RdBu_r', norm=norm)
axes[2].set_title('Predicted Total EP Map')
fig.colorbar(im2, ax=axes[2])
plt.tight_layout()
plt.savefig(f'{current_result_folder}/local_ep_map_2d.png', dpi=150)
plt.show()

cols = min(opt.max_distance + 1, 6)
rows = int(np.ceil((opt.max_distance + 1) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
k_vmax = max(np.abs(pred_map_k).max(), 1e-12)
k_norm = TwoSlopeNorm(vmin=-k_vmax, vcenter=0.0, vmax=k_vmax)
for k in range(opt.max_distance + 1):
    im = axes[k].imshow(pred_map_k[k].T, origin='lower', cmap='RdBu_r', norm=k_norm)
    axes[k].set_title(f'k={k}')
    fig.colorbar(im, ax=axes[k], shrink=0.6)
for i in range(opt.max_distance + 1, len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/local_ep_map_2d_k.png', dpi=150)
plt.show()


In [ ]:
print('[INFO] Calculating Ensemble Averaged Map...')
all_maps = []
test_sampler_batch = CartesianSeqSampler(opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size, device=opt.device, train=False)
with torch.no_grad():
    for batch in tqdm(test_sampler_batch):
        b0 = batch[0].to(test_video.device)
        vslices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        vx = transform(torch.cat(vslices, dim=1).float().to(opt.device))
        m = model(vx, return_maps=True) / opt.scalar
        all_maps.append(m.cpu().numpy())

all_maps = np.concatenate(all_maps, axis=0) / (amb_kwargs['dx']**2 * dt_eff)
ensemble_map_k = all_maps.mean(axis=0)
ensemble_pred_total = ensemble_map_k.sum(axis=0)
ensemble_gt = gt_epr_maps[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
vmax = max(np.abs(ensemble_gt).max(), np.abs(ensemble_pred_total).max(), 1e-12)
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
im0 = axes[0].imshow(ensemble_gt.T, origin='lower', cmap='RdBu_r', norm=norm)
axes[0].set_title('GT Ensemble Mean EPR Map')
fig.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(ensemble_pred_total.T, origin='lower', cmap='RdBu_r', norm=norm)
axes[1].set_title('Predicted Ensemble Total EP Map')
fig.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.savefig(f'{current_result_folder}/ensemble_ep_map_2d.png', dpi=150)
plt.show()

cols = min(opt.max_distance + 1, 6)
rows = int(np.ceil((opt.max_distance + 1) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
spectrum_vmax = max(np.abs(ensemble_map_k).max(), 1e-12)
spectrum_norm = TwoSlopeNorm(vmin=-spectrum_vmax, vcenter=0.0, vmax=spectrum_vmax)
for k in range(opt.max_distance + 1):
    im = axes[k].imshow(ensemble_map_k[k].T, origin='lower', cmap='RdBu_r', norm=spectrum_norm)
    axes[k].set_title(f'k={k}')
    fig.colorbar(im, ax=axes[k], shrink=0.6)
for i in range(opt.max_distance + 1, len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/ensemble_ep_map_2d_k.png', dpi=150)
plt.show()


In [ ]:
model.eval()
all_J = []
test_sampler = CartesianSeqSampler(opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size, device=opt.device, train=False)
with torch.no_grad():
    for batch in test_sampler:
        b0 = batch[0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.cat(slices, dim=1).float().to(opt.device))
        J = model(x) / opt.scalar
        all_J.append(J.cpu().numpy())

all_J = np.concatenate(all_J, axis=0)
mean_J = all_J.mean(axis=0)
std_J = all_J.std(axis=0)
distances = np.arange(0, opt.max_distance + 1)
pred_rate = mean_J * vol / dt_eff
pred_sem = std_J * vol / dt_eff / np.sqrt(len(all_J))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(distances, pred_rate, yerr=pred_sem, capsize=3, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Shell distance k')
axes[0].set_ylabel('<J_k> / dt')
axes[0].set_title('Predicted EP Spectrum')
axes[0].set_xticks(distances)

cum_J = np.cumsum(pred_rate)
axes[1].plot(distances, cum_J, 'o-', color='darkorange')
axes[1].axhline(y=gt_total_epr.mean(), color='black', linestyle='--', label='GT mean total EPR')
axes[1].set_xlabel('Shell distance k')
axes[1].set_ylabel('Cumulative EPR')
axes[1].set_title('Cumulative EP rate')
axes[1].set_xticks(distances)
axes[1].legend()
plt.tight_layout()
plt.savefig(f'{current_result_folder}/ep_spectrum_2d.png', dpi=150)
plt.show()

print(f'Total estimated EPR: {pred_rate.sum():.6e}')
print(f'GT mean total EPR:   {gt_total_epr.mean():.6e}')
for k in range(opt.max_distance + 1):
    print(f'  k={k}: J_k / dt = {pred_rate[k]:.6e} ({100 * mean_J[k] / mean_J.sum():.1f}%)')
